# Stage8 · 自训 feature_reuse subject LoRA:默认模式 vs KV-Cache 主体驱动对比

**问题**:换上我们自己训的 subject LoRA(`independent_condition: true`,QLoRA/NF4,
Subjects200K,512px,ckpt 5000 步)后,`kv_cache=True` 的主体保真度是否达标?

这是 **stage7 的镜像**,补上 2×2 证据矩阵的最后两格:

| LoRA 训练模式 \\ 推理模式 | default(逐步重算) | kvcache(条件 KV 冻结) |
|---|---|---|
| 官方 `subject_512`(default 训) | 主场 ✅(stage7) | 客场,主体丢失 ❌(stage7) |
| 自训 feature_reuse(isolated 训) | 客场,预期略差(**本实验**) | 主场,预期保真 ✅(**本实验**) |

与 stage7 逐字对齐的部分:同 6 张条件图、同 prompt、同 seeds(42/1234)、同 28 步、
同 `image_guidance_scale=1.5`、同 3gpu dispatch。**唯一变量是 LoRA 及其配套 position_delta**:

| | stage7 | stage8(本实验) |
|---|---|---|
| LoRA | 官方 `Yuanshi/OminiControl` subject_512 | `runs/subject_feature_reuse/*/ckpt/5000` |
| 训练注意力 | 默认(条件分支每步看得到主图 latent) | 隔离(`independent_condition: true`) |
| position_delta | 官方约定 `(0, 32)` | **`(0, -32)`**(训练用 `-cond_size//16`,见 train_subject.py) |
| 预期吃亏方 | kvcache | default |

前置:
1. **先停掉训练**(本 notebook 用 cuda:2/3/4,训练占着 8 张卡时跑不了);
2. ckpt 已落盘:`runs/subject_feature_reuse/<时间戳>/ckpt/5000/default.safetensors`。

读图时与 stage7 的产物(`repro/subject_lora_compare_512/`)逐行对照:同一 case 同一 seed,
四张图拼起来就是 2×2 的一整行。

In [1]:
import os
os.chdir("..")  # 仓库根目录(与其它 repro notebook 一致)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys, json, time, glob, random
sys.path.insert(0, "."); sys.path.insert(0, "repro")

import torch
# 与 stage3/probe 相同的兜底:torch 2.8 + diffusers 0.38 的 VAE conv_in 问题
torch.backends.cudnn.enabled = False

from PIL import Image
from kvcache_benchmark import load_pipeline, attach_lora
from omini.pipeline.flux_omini import Condition, generate

# ── 需要改的开关只有这两个 ──────────────────────────────────────────
TARGET_SIZE = 512      # 本 LoRA 只在 512 训过,别改 1024
CKPT_STEP   = 5000     # 想比较 4000/5000 两份 ckpt 时改这里重跑
# ──────────────────────────────────────────────────────────────────
# 自动定位最新一次 feature_reuse 训练的指定步数 ckpt
hits = sorted(
    glob.glob(f"runs/subject_feature_reuse/*/ckpt/{CKPT_STEP}/default.safetensors"),
    key=os.path.getmtime,
)
assert hits, f"找不到 {CKPT_STEP} 步的 ckpt,确认训练已保存到 runs/subject_feature_reuse/*/ckpt/"
CKPT_DIR = os.path.dirname(hits[-1])

COND_SIZE = TARGET_SIZE                  # 主体驱动惯例:条件尺寸 = 目标尺寸
# WHY (0, -32) 而非官方的 (0, 32):必须与训练一致。训练数据管道用的是
# position_delta = [0, -condition_size//16](train_subject.py),符号与官方 512 相反。
POS_DELTA = (0, -COND_SIZE // 16)
STEPS     = 28                           # 与 stage4-7 保持一致,便于横向比较
IMG_GS    = 1.5                          # 与 stage7 同值,保证公平(训练带 10% drop_image,支持 CFG)
SEEDS     = [42, 1234]
OUT_DIR   = f"repro/subject_fr_lora_compare_{TARGET_SIZE}"
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["OUT_DIR"] = OUT_DIR   # 同步给可视化 cell 的 shell magic 用
print(f"[DEBUG] setup: ckpt={CKPT_DIR} delta={POS_DELTA} steps={STEPS} "
      f"img_gs={IMG_GS} cond={COND_SIZE}px target={TARGET_SIZE}px out={OUT_DIR}")

13:56:35 [INFO] cudnn disabled (workaround for VAE CUDNN_STATUS_NOT_INITIALIZED)
13:56:38 [WARNING] Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cu128).
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


[DEBUG] setup: ckpt=runs/subject_feature_reuse/20260716-104006/ckpt/5000 delta=(0, -32) steps=28 img_gs=1.5 cond=512px target=512px out=repro/subject_fr_lora_compare_512


In [2]:
# 强制走本地 HF cache + 直接用 snapshot 路径(双保险,完全绕开网络)。
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

LOCAL_FLUX_PATH = "/home/wuwenxuan03/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev/snapshots/3de623fc3c33e44ffbe2bad470d0f45bccf2eb21"
print("LOCAL_FLUX_PATH exists:", os.path.isdir(LOCAL_FLUX_PATH))

LOCAL_FLUX_PATH exists: True


In [3]:
# 加载 pipeline + 自训 feature_reuse LoRA。
# 注意:推理侧用官方 bf16 全量 + 3gpu dispatch(与 stage7 完全同环境),
# 不用训练时的 NF4 量化 —— LoRA 权重本身是全精度的,挂到什么基座上都一样,
# 而对比实验必须与 stage7 的基座/精度一致,差异才全部归因于 LoRA。
# attach_lora 内部做 set_adapters + device sweep(多卡 dispatch 下 PEFT 会把新 LoRA
# 放错卡,sweep 是必须的,见 kvcache_benchmark.py 失败 #11 注释)。
pipe = load_pipeline(LOCAL_FLUX_PATH, dispatch="3gpu", gpu0=2, gpu1=3, gpu2=4)
attach_lora(pipe, CKPT_DIR, "default.safetensors", "feature_reuse")

13:56:47 [INFO] GPU 0: NVIDIA GeForce RTX 4090 | 23.6 GB
13:56:47 [INFO] GPU 1: NVIDIA GeForce RTX 4090 | 23.6 GB
13:56:47 [INFO] GPU 2: NVIDIA GeForce RTX 4090 | 23.6 GB
13:56:47 [INFO] GPU 3: NVIDIA GeForce RTX 4090 | 23.6 GB
13:56:47 [INFO] GPU 4: NVIDIA GeForce RTX 4090 | 23.6 GB
13:56:47 [INFO] GPU 5: NVIDIA GeForce RTX 4090 | 23.6 GB
13:56:47 [INFO] GPU 6: NVIDIA GeForce RTX 4090 | 23.6 GB
13:56:47 [INFO] GPU 7: NVIDIA GeForce RTX 4090 | 23.6 GB
13:56:47 [INFO] 加载 FLUX pipeline: /home/wuwenxuan03/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev/snapshots/3de623fc3c33e44ffbe2bad470d0f45bccf2eb21 (bf16, 3gpu 手工 dispatch)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
13:58:28 [INFO] dispatch 完成: cuda:2 = encoders+vae+latents(主场), cuda:3 = embedders+dual blocks, cuda:4 = single blocks+out
13:58:28 [INFO] 已安装 tx bridge:输入 -> transformer 卡,输出 -> 主场卡
13:58:28 [INFO] 加载 LoRA: runs/subject_feature_reuse/20260716-104006/ckpt/5000 :: default.safetensors
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
13:58:29 [INFO] device 安全网:搬了 0 个 param/buffer


FluxPipeline {
  "_class_name": "FluxPipeline",
  "_diffusers_version": "0.38.0",
  "_name_or_path": "/home/wuwenxuan03/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev/snapshots/3de623fc3c33e44ffbe2bad470d0f45bccf2eb21",
  "feature_extractor": [
    null,
    null
  ],
  "image_encoder": [
    null,
    null
  ],
  "scheduler": [
    "diffusers",
    "FlowMatchEulerDiscreteScheduler"
  ],
  "text_encoder": [
    "transformers",
    "CLIPTextModel"
  ],
  "text_encoder_2": [
    "transformers",
    "T5EncoderModel"
  ],
  "tokenizer": [
    "transformers",
    "CLIPTokenizer"
  ],
  "tokenizer_2": [
    "transformers",
    "T5TokenizerFast"
  ],
  "transformer": [
    "diffusers",
    "FluxTransformer2DModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKL"
  ]
}

In [4]:
# 6 张主体条件图,与 stage6/stage7 完全同一组(便于三个实验逐行对照)。
# 前 5 条 prompt 逐字取自官方 examples/subject*.ipynb;book 自拟(官方无 subject 用例)。
CASES = [
    ("penguin", "assets/penguin.jpg",
     "On Christmas evening, on a crowded sidewalk, this item sits on the road, "
     "covered in snow and wearing a Christmas hat."),
    ("rc_car", "assets/rc_car.jpg",
     "A film style shot. On the moon, this item drives across the moon surface. "
     "The background is that Earth looms large in the foreground."),
    ("tshirt", "assets/tshirt.jpg",
     "On the beach, a lady sits under a beach umbrella. She's wearing this shirt and has "
     "a big smile on her face, with her surfboard hehind her. The sun is setting in the "
     "background. The sky is a beautiful shade of orange and purple."),
    ("clock", "assets/clock.jpg",
     "In a Bauhaus style room, this item is placed on a shiny glass table, with a vase of "
     "flowers next to it. In the afternoon sun, the shadows of the blinds are cast on the wall."),
    ("oranges", "assets/oranges.jpg",
     "A very close up view of this item. It is placed on a wooden table. The background is "
     "a dark room, the TV is on, and the screen is showing a cooking show."),
    ("book", "assets/book.jpg",  # prompt 自拟(官方无 subject 用例)
     "This item is placed on a wooden desk in a library, next to a cup of coffee, "
     "under the warm light of a reading lamp."),
]

def build_condition(image_path):
    # 官方预处理:center-crop 成正方形再 resize(pipeline 不会自动做)
    img = Image.open(image_path).convert("RGB")
    w, h = img.size; m = min(w, h)
    img = img.crop(((w - m) // 2, (h - m) // 2, (w + m) // 2, (h + m) // 2))
    img = img.resize((COND_SIZE, COND_SIZE))
    # 与 stage7 唯一的分支差异:adapter="feature_reuse" → 条件分支走自训 LoRA
    # (文本/主图分支仍走基座,与训练时 adapter_names=[None, None, "default"] 一致)
    return img, Condition(img, "feature_reuse", position_delta=POS_DELTA)

def run_one(prompt, condition, seed, kv_cache):
    # WHY 每次重建 generator:两种模式必须从同一噪声出发,差异才全部归因于注意力模式
    g = torch.Generator(device="cuda:0").manual_seed(seed)
    t0 = time.perf_counter()
    out = generate(pipe, prompt=prompt, conditions=[condition],
                   height=TARGET_SIZE, width=TARGET_SIZE,
                   num_inference_steps=STEPS, guidance_scale=3.5,
                   image_guidance_scale=IMG_GS,   # 与 stage7 同值,保持公平
                   generator=g, kv_cache=kv_cache)
    return out.images[0], time.perf_counter() - t0

In [5]:
# 主循环:6 case × 2 seed × 2 模式 = 24 次生成
records = []
for name, path, prompt in CASES:
    cond_img, condition = build_condition(path)
    cond_img.save(f"{OUT_DIR}/{name}_cond.png")
    for seed in SEEDS:
        row = {"case": name, "seed": seed, "prompt": prompt,
               "ckpt": CKPT_DIR}   # 与 stage7 records 的差异:记下用的哪份 ckpt
        for mode, kv in (("default", False), ("kvcache", True)):
            img, dt = run_one(prompt, condition, seed, kv)
            fp = f"{OUT_DIR}/{name}_s{seed}_{mode}.png"
            img.save(fp)
            row[mode] = fp; row[f"{mode}_sec"] = round(dt, 1)
        records.append(row)
        print(f"[DEBUG] run_one: {name} seed={seed} "
              f"default={row['default_sec']}s kvcache={row['kvcache_sec']}s "
              f"speedup={row['default_sec']/row['kvcache_sec']:.2f}x")

with open(f"{OUT_DIR}/records.json", "w") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"完成 {len(records)} 组对照,已写 {OUT_DIR}/records.json")

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

[DEBUG] run_one: penguin seed=42 default=20.1s kvcache=12.0s speedup=1.68x


  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

[DEBUG] run_one: book seed=42 default=19.5s kvcache=12.0s speedup=1.62x


  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

[DEBUG] run_one: book seed=1234 default=19.5s kvcache=12.0s speedup=1.62x
完成 12 组对照,已写 repro/subject_fr_lora_compare_512/records.json


In [6]:
# 可视化:复用独立脚本,产物 $OUT_DIR/comparison_grid.png
!python repro/visualize_quality_compare.py \
    --records $OUT_DIR/records.json \
    --out-dir $OUT_DIR

[INFO] repo_root  : /home/wuwenxuan03/OminiControl
[INFO] records    : repro/subject_fr_lora_compare_512/records.json
[INFO] out_dir    : repro/subject_fr_lora_compare_512
[INFO] 已生成     : repro/subject_fr_lora_compare_512/comparison_grid.png  (5997 KB)

对比摘要  (12 组, 同 LoRA 同 seed)
  penguin  s  42  default= 20.1s  kvcache= 12.0s  speedup=1.68x
  penguin  s1234  default= 19.5s  kvcache= 12.0s  speedup=1.62x
  rc_car   s  42  default= 19.5s  kvcache= 12.0s  speedup=1.62x
  rc_car   s1234  default= 19.6s  kvcache= 12.0s  speedup=1.63x
  tshirt   s  42  default= 19.5s  kvcache= 12.0s  speedup=1.62x
  tshirt   s1234  default= 19.5s  kvcache= 12.0s  speedup=1.62x
  clock    s  42  default= 19.5s  kvcache= 12.0s  speedup=1.62x
  clock    s1234  default= 19.5s  kvcache= 12.0s  speedup=1.62x
  oranges  s  42  default= 19.5s  kvcache= 12.0s  speedup=1.62x
  oranges  s1234  default= 19.5s  kvcache= 12.0s  speedup=1.62x
  book     s  42  default= 19.5s  kvcache= 12.0s  speedup=1.62x
  book     s1

In [ ]:
# GSB 标注骨架(格式对齐 scripts/compute_gsb.py),盲评随机换边
rng = random.Random(0)
annotations, side_map = {}, {}
for r in records:
    cid = f"{r['case']}_s{r['seed']}"
    flip = rng.random() < 0.5
    side_map[cid] = {"A": "kvcache" if flip else "default",
                     "B": "default" if flip else "kvcache"}
    annotations[cid] = {"prompt": r["prompt"], "choices": {"quality": ""}}

with open(f"{OUT_DIR}/annotations.json", "w") as f:
    json.dump({"annotations": annotations}, f, ensure_ascii=False, indent=2)
with open(f"{OUT_DIR}/side_map.json", "w") as f:
    json.dump(side_map, f, ensure_ascii=False, indent=2)
print(f"标注后统计: python scripts/compute_gsb.py {OUT_DIR}/annotations.json")

## 读图指引

1. **第一观察点:kvcache 侧的主体保真度**。stage7 里官方 LoRA 开缓存后主体身份大面积丢失
   (penguin 最惨);如果本实验 kvcache 侧主体稳(纹理、配色、结构贴条件图),
   复现的核心命题——"隔离训练使条件特征与 timestep 无关,缓存推理与逐步重算等价"——成立。
2. **第二观察点:default 侧掉多少**。条件分支按隔离模式训,default 推理时它却能被主图
   attend 到每一步,对这块 LoRA 是分布外——预期轻微吃亏(stage4 在 256px 的 fill 任务上
   已见此方向)。掉得少说明隔离训练代价小,是加分项。
3. **横向对照**:与 `repro/subject_lora_compare_512/` 同名文件逐张比。四张图
   (stage7 default / stage7 kvcache / stage8 default / stage8 kvcache)拼成 2×2 的一行,
   期望的对角线格局:各自主场好、客场差,且 stage8-kvcache ≫ stage7-kvcache。
4. **公平性注意**:本 LoRA 只训了 5000 步(等效 ~4 万张图),官方 LoRA 训练量约为其十倍,
   绝对画质(细节、光影)可能整体不如官方——所以结论只看【模式间相对差】,
   不拿 stage8 的绝对画质与 stage7 比高下。
5. **速度**:与 stage7 相同的 ~1.6x speedup 预期(条件 token 恒占序列一半,
   `image_guidance_scale=1.5` 的空条件 CFG 分支两模式同加)。

标注 `annotations.json` 的 `choices.quality`(G = default 好,B = kvcache 好,S = 持平,
按 `side_map.json` 还原盲评换边)后运行 `python scripts/compute_gsb.py` 统计。

**跑完后的 2×2 证据矩阵读法**:stage7 已证 "default 训的 LoRA 开缓存会坏";本实验若证
"隔离训的 LoRA 开缓存不坏",则 KV-Cache 的工程价值(1.6x 加速)就有了无损质量的前提,
OminiControl2 feature reuse 的复现闭环完成。